<a href="https://colab.research.google.com/github/Sadaf-001/ncua-call-report-processing/blob/main/NCUA_Call_Report_Data_Processing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
import os
import zipfile
import pandas as pd
from functools import reduce

# =========================
# CREATE FOLDERS
# =========================

os.makedirs("raw", exist_ok=True)
os.makedirs("csv_output", exist_ok=True)
os.makedirs("merged_output", exist_ok=True)

# =========================
# EXTRACT ZIP
# =========================

zip_path = "call-report-data-2025-03.zip"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall("raw")

print("ZIP extracted.")

# =========================
# FILES TO CONVERT
# =========================

required_files = [
    "Credit Union Branch Information.txt",
    "FS220.txt",
    "FS220A.txt",
    "FS220B.txt",
    "FS220C.txt",
    "FS220D.txt",
    "FS220G.txt",
    "FS220H.txt",
    "FS220I.txt",
    "FS220J.txt",
    "FS220K.txt",
    "FS220L.txt",
    "FS220M.txt",
    "FS220N.txt",
    "FS220P.txt",
    "FS220Q.txt",
    "FS220R.txt",
    "FS220S.txt"
]

# =========================
# TXT TO CSV
# =========================

for file_name in required_files:

    txt_path = os.path.join("raw", file_name)

    try:
        df = pd.read_csv(
            txt_path,
            encoding='latin1',
            low_memory=False,
            on_bad_lines='skip'
        )

        # clean column names
        df.columns = df.columns.str.strip()

        csv_name = file_name.replace(".txt", ".csv")

        csv_path = os.path.join("csv_output", csv_name)

        df.to_csv(csv_path, index=False)

        print(f"Converted: {csv_name}")

    except Exception as e:
        print(f"Error with {file_name}: {e}")
# =========================
# MERGE CSV FILES
# =========================

# =========================
# MERGE CSV FILES
# =========================

merge_files = [
    "FS220.csv",
    "FS220A.csv",
    "FS220B.csv",
    "FS220C.csv",
    "FS220D.csv",
    "FS220G.csv",
    "FS220H.csv",
    "FS220I.csv",
    "FS220J.csv",
    "FS220K.csv",
    "FS220L.csv",
    "FS220M.csv",
    "FS220N.csv",
    "FS220P.csv",
    "FS220Q.csv",
    "FS220R.csv",
    "FS220S.csv"
]

merge_keys = ["CU_NUMBER", "CYCLE_DATE", "JOIN_NUMBER"]

dfs = []

for file_name in merge_files:

    path = os.path.join("csv_output", file_name)

    try:
        df = pd.read_csv(path, low_memory=False)

        # remove spaces from column names
        df.columns = df.columns.str.strip()

        print(f"\nColumns in {file_name}:")
        print(df.columns.tolist())

        # check if required merge columns exist
        missing_cols = [col for col in merge_keys if col not in df.columns]

        if missing_cols:
            print(f"Skipping {file_name} because missing columns: {missing_cols}")
            continue

        dfs.append(df)

        print(f"Loaded {file_name}")

    except Exception as e:
        print(f"Error loading {file_name}: {e}")

# make sure we actually have files to merge
if len(dfs) == 0:
    print("No valid files found for merging.")

else:

    merged_df = reduce(
        lambda left, right:
        pd.merge(
            left,
            right,
            on=merge_keys,
            how='outer'
        ),
        dfs
    )

    merged_df.to_csv(
        "merged_output/merged_file.csv",
        index=False
    )

    print("MERGE COMPLETE")

ZIP extracted.
Converted: Credit Union Branch Information.csv
Converted: FS220.csv
Converted: FS220A.csv
Converted: FS220B.csv
Converted: FS220C.csv
Converted: FS220D.csv
Converted: FS220G.csv
Converted: FS220H.csv
Converted: FS220I.csv
Converted: FS220J.csv
Converted: FS220K.csv
Converted: FS220L.csv
Converted: FS220M.csv
Converted: FS220N.csv
Converted: FS220P.csv
Converted: FS220Q.csv
Converted: FS220R.csv
Converted: FS220S.csv

Columns in FS220.csv:
['CU_NUMBER', 'CYCLE_DATE', 'JOIN_NUMBER', 'UPDATE_DATE', 'ACCT_007', 'ACCT_008', 'ACCT_009', 'ACCT_010', 'ACCT_011C', 'ACCT_013', 'ACCT_018', 'ACCT_019', 'ACCT_021B', 'ACCT_022B', 'ACCT_023B', 'ACCT_025A', 'ACCT_025B', 'ACCT_031A', 'ACCT_031B', 'ACCT_041B', 'ACCT_042', 'ACCT_058C', 'ACCT_065', 'ACCT_067', 'ACCT_068', 'ACCT_069', 'ACCT_080', 'ACCT_083', 'ACCT_084', 'ACCT_100', 'ACCT_300', 'ACCT_340', 'ACCT_380', 'ACCT_386', 'ACCT_387', 'ACCT_388', 'ACCT_392', 'ACCT_400', 'ACCT_457', 'ACCT_521', 'ACCT_522', 'ACCT_523', 'ACCT_524', 'ACCT_

In [7]:
# =========================
# FINAL MERGE
# =========================

from functools import reduce

merged_df = reduce(
    lambda left, right:
    pd.merge(
        left,
        right,
        on=["CU_NUMBER", "CYCLE_DATE", "JOIN_NUMBER"],
        how="outer"
    ),
    dfs
)

print("Merge successful!")

print(merged_df.shape)

merged_df.to_csv(
    "merged_output/merged_file.csv",
    index=False
)

print("merged_file.csv saved successfully!")

Merge successful!
(4505, 2687)
merged_file.csv saved successfully!
